In [1]:
%matplotlib inline

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import optuna
import xgboost as xgb
import catboost as cb
from sklearn.isotonic import IsotonicRegression

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.feature_pipeline import FeaturePipeline
from src.pipeline.artifacts import default_feature_artifacts
from src.pipeline.data_preparation import load_and_prepare_data
from src.models.train_balanced_bagging_ensemble import run_balanced_bagging
from src.features.extract import load_feature_config, feature_config_hash_text
from src.optimization.firewall_objective import compute_firewall_score

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, auc

paths = load_paths()
logger = setup_logger(level="INFO")

features_yaml = paths.configs_dir / "features.yaml"


C:\Users\scoti\PycharmProjects\ai-vpn-firewall\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. Load Data
logger.info("Loading and preparing data...")
df_all = load_and_prepare_data()


2026-03-09 22:56:44 | INFO | ai-vpn-firewall | Loading and preparing data...
2026-03-09 22:56:44 | INFO | ai-vpn-firewall | Loading and processing VNAT...
2026-03-09 22:57:15 | INFO | ai-vpn-firewall | Loading and processing ISCX...
2026-03-09 22:58:06 | INFO | ai-vpn-firewall | Data loaded. Shape: (19908, 45)
2026-03-09 22:58:06 | INFO | ai-vpn-firewall | Split counts:
split
train    16138
val       1926
test      1844
Name: count, dtype: int64


In [3]:
# 2. Fit Feature Pipeline
logger.info("Fitting FeaturePipeline...")
pipeline = FeaturePipeline().fit(df_all[df_all["split"] == "train"])

feature_art = default_feature_artifacts(paths.artifacts_dir / "features")
pipeline.save(feature_art, feature_config_hash=feature_config_hash_text(features_yaml))

feature_cols = pipeline.model_feature_names()
logger.info(f"Pipeline saved. Model features: {len(feature_cols)}")


2026-03-09 22:58:07 | INFO | ai-vpn-firewall | Fitting FeaturePipeline...
2026-03-09 22:58:07 | INFO | ai-vpn-firewall | Pipeline saved. Model features: 39


In [4]:
# 3. Transform Data
logger.info("Transforming features...")
df_transformed = pipeline.transform(df_all)

# Add metadata back
meta_cols = ["label", "split", "capture_id", "dataset", "flow_id"]
for col in meta_cols:
    df_transformed[col] = df_all[col].values

print("Transformed shape:", df_transformed.shape)


2026-03-09 22:58:07 | INFO | ai-vpn-firewall | Transforming features...
Transformed shape: (19908, 45)


In [5]:
# 4. Prepare Splits for Optimization
train_df = df_transformed[df_transformed["split"] == "train"]
val_df = df_transformed[df_transformed["split"] == "val"]

X_train = train_df[feature_cols].values
y_train = train_df["label"].values

X_val = val_df[feature_cols].values
y_val = val_df["label"].values
val_groups = val_df["capture_id"].values


In [6]:
# 5. XGBoost Optimization
def xgb_objective(trial: optuna.Trial):
    params = {
        "objective": "binary:logistic", "eval_metric": "logloss", "booster": "gbtree",
        "tree_method": "hist", "n_estimators": 1000, "random_state": 42, "n_jobs": 1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 100.0, log=True),
    }
    model = xgb.XGBClassifier(**params, early_stopping_rounds=150)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    p_val_raw = model.predict_proba(X_val, iteration_range=(0, model.best_iteration + 1))[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_val_raw, y_val)
    p_val_calib = iso.transform(p_val_raw)
    val_res = pd.DataFrame({"capture_id": val_groups, "label": y_val, "prob": p_val_calib})
    return compute_firewall_score(val_res)

logger.info("Starting XGBoost optimization...")
xgb_study = optuna.create_study(direction="maximize")
xgb_study.optimize(xgb_objective, n_trials=500) # Full optimization
xgb_params = xgb_study.best_params
print("Best XGBoost Params:", xgb_params)


2026-03-09 22:58:07 | INFO | ai-vpn-firewall | Starting XGBoost optimization...


[I 2026-03-09 22:58:07,518] A new study created in memory with name: no-name-919396f6-7c84-4d83-b6b8-a972a569a453
[I 2026-03-09 22:58:10,940] Trial 0 finished with value: 97.02310924369748 and parameters: {'learning_rate': 0.06198218499247607, 'max_depth': 7, 'subsample': 0.9046447422469218, 'colsample_bytree': 0.7653142494900714, 'min_child_weight': 1, 'reg_alpha': 7.29454145598956, 'reg_lambda': 4.533259960550638, 'scale_pos_weight': 10.646988642289136}. Best is trial 0 with value: 97.02310924369748.
[I 2026-03-09 22:58:13,555] Trial 1 finished with value: 90.55252100840335 and parameters: {'learning_rate': 0.047263046895485064, 'max_depth': 9, 'subsample': 0.8736201398616024, 'colsample_bytree': 0.7996138632909295, 'min_child_weight': 5, 'reg_alpha': 9.512627476088822, 'reg_lambda': 1.5119544193404433, 'scale_pos_weight': 16.926826420417513}. Best is trial 0 with value: 97.02310924369748.
[I 2026-03-09 22:58:15,246] Trial 2 finished with value: 84.08193277310923 and parameters: {'le

Best XGBoost Params: {'learning_rate': 0.04663772558780211, 'max_depth': 5, 'subsample': 0.9680210105482272, 'colsample_bytree': 0.7353295330857288, 'min_child_weight': 9, 'reg_alpha': 1.0417924997757333, 'reg_lambda': 0.5510630408233896, 'scale_pos_weight': 28.380957680592648}


In [7]:
# 6. CatBoost Optimization
def cat_objective(trial: optuna.Trial):
    params = {
        "iterations": 1000, "random_seed": 42, "thread_count": 1,
        "verbose": False, "allow_writing_files": False, "early_stopping_rounds": 150,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 100.0, log=True),
    }
    model = cb.CatBoostClassifier(**params)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)
    p_val_raw = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip").fit(p_val_raw, y_val)
    p_val_calib = iso.transform(p_val_raw)
    val_res = pd.DataFrame({"capture_id": val_groups, "label": y_val, "prob": p_val_calib})
    return compute_firewall_score(val_res)

logger.info("Starting CatBoost optimization...")
cat_study = optuna.create_study(direction="maximize")
cat_study.optimize(cat_objective, n_trials=500) # Full optimization
cat_params = cat_study.best_params
print("Best CatBoost Params:", cat_params)


2026-03-09 23:13:57 | INFO | ai-vpn-firewall | Starting CatBoost optimization...


[I 2026-03-09 23:13:58,001] A new study created in memory with name: no-name-620bb756-7699-4d58-b870-b26a346d6b8c
[I 2026-03-09 23:14:31,347] Trial 0 finished with value: 88.7878151260504 and parameters: {'learning_rate': 0.032513619959075306, 'depth': 9, 'l2_leaf_reg': 3.3813251350964433, 'random_strength': 0.26493163819617, 'bagging_temperature': 0.4912178700043367, 'border_count': 223, 'scale_pos_weight': 2.479005384140389}. Best is trial 0 with value: 88.7878151260504.
[I 2026-03-09 23:14:35,612] Trial 1 finished with value: 103.49369747899158 and parameters: {'learning_rate': 0.15363204604585093, 'depth': 7, 'l2_leaf_reg': 3.1733281196982905, 'random_strength': 0.009838969797145582, 'bagging_temperature': 0.6375820542945706, 'border_count': 104, 'scale_pos_weight': 3.7145746927478926}. Best is trial 1 with value: 103.49369747899158.
[I 2026-03-09 23:14:44,648] Trial 2 finished with value: 97.02310924369748 and parameters: {'learning_rate': 0.11241167761476732, 'depth': 8, 'l2_leaf

Best CatBoost Params: {'learning_rate': 0.15363204604585093, 'depth': 7, 'l2_leaf_reg': 3.1733281196982905, 'random_strength': 0.009838969797145582, 'bagging_temperature': 0.6375820542945706, 'border_count': 104, 'scale_pos_weight': 3.7145746927478926}


In [8]:
# 7. Train Ensemble with Tuned Models
logger.info("Training Tuned Ensemble...")
output_dir = paths.artifacts_dir / "balanced_bagging_tuned"
results = run_balanced_bagging(
    df=df_transformed,
    label_col="label", group_col="capture_id", dataset_col="dataset", split_col="split",
    bags_per_family=3, majority_ratio=1.0, target_fprs="0.001,0.005,0.01", seed=42,
    output_dir=str(output_dir), model_types=["xgb", "lgbm", "cat"], feature_cols=feature_cols,
    weight_xgb=1.0, weight_lgbm=1.0, weight_cat=1.0,
    xgb_params=xgb_params,
    cat_params=cat_params
)


2026-03-10 00:26:08 | INFO | ai-vpn-firewall | Training Tuned Ensemble...


In [9]:
# 8. Load Predictions for Evaluation
preds_path = output_dir / "predictions.csv"
df_preds = pd.read_csv(preds_path)
print(f"Loaded {len(df_preds)} predictions.")


Loaded 19908 predictions.


In [10]:
# 9. Session-Level Firewall Evaluation
print("\n" + "="*60)
print("SESSION FIREWALL EVALUATION (TUNED)")
print("="*60)

def tune_firewall_thresholds(df_val_sess, score_col="score_mean", label_col="label"):
    y_true = df_val_sess[label_col].astype(int)
    scores = df_val_sess[score_col].values
    desc_idxs = np.argsort(scores)[::-1]
    y_sorted, s_sorted = y_true.values[desc_idxs], scores[desc_idxs]
    fps = np.cumsum(1 - y_sorted)
    idx_zero = np.searchsorted(fps, 1, side='left') - 1
    block_thr = s_sorted[idx_zero] if idx_zero >= 0 else s_sorted[0] + 1e-6
    idx_one = np.searchsorted(fps, 2, side='left') - 1
    monitor_thr = s_sorted[idx_one] if idx_one >= 0 else s_sorted[0] + 1e-6
    if monitor_thr > block_thr: monitor_thr = block_thr
    return block_thr, monitor_thr

def apply_firewall_policy(df_sess, block_thr, monitor_thr, score_col="score_mean"):
    def classify(score):
        if score >= block_thr: return "BLOCK"
        if score >= monitor_thr: return "MONITOR"
        return "ALLOW"
    return df_sess[score_col].apply(classify)

def report_firewall_metrics(df_sess, policy_col="policy", label_col="label", title=""):
    n_benign = (df_sess[label_col] == 0).sum()
    n_vpn = (df_sess[label_col] == 1).sum()
    is_block = df_sess[policy_col] == "BLOCK"
    is_flagged = df_sess[policy_col].isin(["BLOCK", "MONITOR"])
    tp_block = (is_block & (df_sess[label_col] == 1)).sum()
    fp_block = (is_block & (df_sess[label_col] == 0)).sum()
    tp_flagged = (is_flagged & (df_sess[label_col] == 1)).sum()
    fp_flagged = (is_flagged & (df_sess[label_col] == 0)).sum()
    block_recall = tp_block / n_vpn if n_vpn > 0 else 0
    flagged_recall = tp_flagged / n_vpn if n_vpn > 0 else 0
    print(f"\n--- {title} ---")
    print(f"BLOCK Recall:   {block_recall:.4f} (TP={tp_block}/{n_vpn})")
    print(f"FLAGGED Recall: {flagged_recall:.4f} (TP={tp_flagged}/{n_vpn})")
    return {"block_recall": block_recall, "flagged_recall": flagged_recall}

val_flows = df_preds[df_preds["split"] == "val"].copy()
test_flows = df_preds[df_preds["split"] == "test"].copy()
agg_funcs = {"label": "max", "prob_iso": "mean"}
val_sess = val_flows.groupby(["capture_id", "dataset"]).agg(agg_funcs).rename(columns={"prob_iso": "score_mean"})
test_sess = test_flows.groupby(["capture_id", "dataset"]).agg(agg_funcs).rename(columns={"prob_iso": "score_mean"})
b_thr, m_thr = tune_firewall_thresholds(val_sess)
print(f"\nTuned Thresholds (Val): BLOCK >= {b_thr:.4f}, MONITOR >= {m_thr:.4f}")
test_sess["policy"] = apply_firewall_policy(test_sess, b_thr, m_thr)
metrics = report_firewall_metrics(test_sess, title="Test Set Evaluation")



SESSION FIREWALL EVALUATION (TUNED)

Tuned Thresholds (Val): BLOCK >= 0.9135, MONITOR >= 0.6707

--- Test Set Evaluation ---
BLOCK Recall:   0.6471 (TP=11/17)
FLAGGED Recall: 0.9412 (TP=16/17)


In [11]:
# 10. Comparison with Baseline
print("\n" + "="*60)
print("COMPARISON WITH BASELINE")
print("="*60)

baseline_metrics = {"block_recall": 0.5882, "flagged_recall": 0.9412}
print(f"{'Metric':<20} | {'Baseline':<10} | {'Tuned':<10} | {'Diff':<10}")
print("-" * 56)
for k in ["block_recall", "flagged_recall"]:
    base, curr = baseline_metrics.get(k, 0), metrics.get(k, 0)
    print(f"{k:<20} | {base:.4f}     | {curr:.4f}     | {curr - base:+.4f}")

print("\nDone.")



COMPARISON WITH BASELINE
Metric               | Baseline   | Tuned      | Diff      
--------------------------------------------------------
block_recall         | 0.5882     | 0.6471     | +0.0589
flagged_recall       | 0.9412     | 0.9412     | -0.0000

Done.
